In [4]:
import json
from sentence_transformers import SentenceTransformer
import numpy as np

In [3]:
with open("../output/RO_NECP_chunks.json", encoding="utf-8") as f:
    chunks = json.load(f)

model = SentenceTransformer("BAAI/bge-small-en-v1.5")

# chunk -> vector
texts=[c["text"] for c in chunks]
vectors=model.encode(texts, normalize_embeddings=True, show_progress_bar=True)

print(vectors.shape)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/20 [00:00<?, ?it/s]

(636, 384)


In [6]:
question = "How much CO2 will Romania emit in 2030?"
q = model.encode("Represent this sentence for searching relevant passages: " + question, normalize_embeddings=True)

scores = vectors @ q
top5 = np.argsort(scores)[::-1][:5]

for i in top5:
    print(f"page {chunks[i]['page']}  score {scores[i]:.3f}")
    print(chunks[i]["text"][:200], "\n")

page 49  score 0.791
Emissions from non-ETS economic activities for the period 2021-2030 are established in Regulation (EU) 2018/842 of the European Parliament and of the Council of 30 May 2018 on binding annual greenhous 

page 65  score 0.763
of 0.83 Mtoe in the final consumption in 2030 compared to the baseline scenario. All of the three scenarios lead to final energy consumption of 3.38 Mtoe in buildings in 2050, which is a 66 % reductio 

page 170  score 0.757
Source: EU Commission, Energy datasheets – EU 28, 8 October 2019 RO EN Ţiţei şi produse petroliere Crude oil and petroleum products Gaz natural Natural gas Energie regenerabilă Renewable energy Electr 

page 141  score 0.747
B. Analytical basis55 4. Current situation and projections with existing policies and measures 4.1. Projected evolution of main exogenous factors influencing energy system and GHG emission development 

page 48  score 0.746
based on Romania’s report on the GHG emission projections issued by the European 

### testing

In [7]:
questions=[
"How much geothermal energy will be used in district heating in 2030?", # od anex2, table7
"How much will renewable energy in buildings increase by 2030?", # anex2,table7
"What does the NECP say about renewable energy communities?",  #anex2, table7
"What are the expected primary and final energy savings in 2025?",  # chart 10, anex4, table1
"What are the primary and final energy savings targets for 2030?", # chart 10
"What will primary energy consumption be in 2030?",
"Which law requires local authorities to separately collect municipal waste?" #page83
]

In [8]:
def search(question):
    q=model.encode("Represent this sentence for searching relevant passages: " + question, normalize_embeddings=True)

    scores=vectors @ q
    top = np.argsort(scores)[::-1][:5]
    print(f"\n=== {question}")
    for i in top:
        print(f"page {chunks[i]['page']}  score {scores[i]:.3f}  |  {chunks[i]['text'][:150]}")


In [9]:
for q in questions:
    search(q)


=== How much geothermal energy will be used in district heating in 2030?
page 60  score 0.724  |  The biomass potential in agriculture, which mainly consists of stems, including maize cobs, was estimated between 21.5 and 35.8 million tonnes, with t
page 143  score 0.716  |  Moreover, MDRAP (the current MLPDA) is currently implementing the “District Heating” Programme targeted to district heating systems for localities. Th
page 164  score 0.707  |  Chart 31 - Trend in the structure of final energy consumption by energy source in the period 2006- 2017 Source: EU Commission, Energy datasheets – EU 
page 58  score 0.699  |  by 2027-2030, in addition to deploying additional wind and solar capacities, it will be necessary to keep the existing capacity by repowering. In this
page 5  score 0.688  |  Projected trend, in the context of current policies, in the gross final consumption and consumption of energy from renewable sources for heating and c

=== How much will renewable energy in buildi

In [10]:
np.save("../output/RO_NECP_vectors.npy", vectors)